# Super TikTakToe: two short tests before the final model, on Google Colab

Both tests continue the finished **large** network (iteration 240) exactly like the earlier test `more_games`
(1024 games per iteration, every position used 2 times), with one more change that gives **more games for the
same computing time**:

| `TEST` | What changes |
|---|---|
| `fast_moves` | KataGo's trick: the full search (400 simulations) on a quarter of the moves, 100 on the others; only the full ones are learned from |
| `sims_200` | 200 simulations on every move |

Each test runs for **1.56 hours of self-play and learning** (the time `more_games`' 20 iterations take today, guessed)
and then stops by itself. Afterwards its network plays a direct match against `more_games`' network (iteration 260):
256 games, 500 simulations. **Rule set beforehand: 53% or more, and the test's setting is used for the final run.**

1. Open this notebook twice, in two Colab sessions, and set `TEST` in the first code cell, once to `fast_moves` and once to `sims_200`.
2. Menu **Runtime > Change runtime type**: choose the L4.
3. Run the cells from top to bottom. The speed check runs only in the `fast_moves` session (about 10 minutes).
4. The first start copies the network and its memory into `Super-TikTakToe-models/experiments/<TEST>` (a few minutes)
   and plays large's last 4 checks again with 512 games (the baseline). Every 4th line has the check against medium.
5. Guessed: about 2 hours and 4 units per session, about 8 units for both.
6. After a disconnect run all cells again: the test continues and still stops at the same time limit.
7. When it has ended: **Runtime > Disconnect and delete runtime**. Send me the output of the last two cells.


In [ ]:
# Which test this session runs: 'fast_moves' or 'sims_200'
TEST = 'fast_moves'

# Google Drive: the folder with the finished networks (large, medium, experiments/more_games)
from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/Super-TikTakToe-models'
import os
os.environ['SUPER_TTT_MODELS'] = MODELS
assert os.path.exists(f'{MODELS}/experiments/more_games/alphazero_large.pt'), 'more_games network missing'

In [ ]:
# The code from GitHub (the newest version every time)
%cd /content
!test -d Super-TikTakToe || git clone https://github.com/kasparhui/Super-TikTakToe.git
%cd /content/Super-TikTakToe
!git pull

In [ ]:
# Which graphics card did this session get?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Speed of large with the search on 1 core and on all cores, 512 and 1024 games at once (only in the fast_moves session)
if TEST == 'fast_moves':
    !python -u -c "import sys; sys.path.insert(0, 'src'); import pretest as P; P.BOTTLENECK_PARALLEL_GAMES = (512, 1024); P.bottleneck('large')"

In [ ]:
# The test. Every line is one iteration and shows the time used of the 1.56 hours.
# Stop with the stop button of this cell: everything is saved, and running it again continues.
!python -u src/train.py experiment {TEST}

In [ ]:
# The direct match against more_games' network (iteration 260): 256 games, 500 simulations
!python -u src/train.py match {TEST} more_games

In [ ]:
# The checks against medium (512 games); iterations up to 240 are the finished large network
import pandas as pd
log = pd.read_csv(f'{MODELS}/experiments/{TEST}/alphazero_large_log.csv')
checks = log[log['vs_medium'].notna()][['iteration', 'games', 'positions', 'simulations', 'vs_medium', 'wins_medium', 'ties_medium', 'losses_medium', 'work_seconds', 'seconds']]
checks.tail(16)